# My AI Resume/ Job description Analyser

# STAGE 1

### Code for PDF text extraction

In [ ]:
from extraction.pipeline import process_pdf

text = process_pdf("Downloads/My_Resume(9).pdf")
print(text)
print(f"\n\nTotal characters extracted: {len(text)}")

### Code for Docx text extraction

In [ ]:
from extraction.docx_reader import extract_text_from_docx

text = extract_text_from_docx("Downloads/Onyeiwu_Gabriel_Chibuzor_Graduate_Trainee_CV.docx")
print(text)
print(f"\n\nTotal characters extracted: {len(text)}")

### Code for PDF and Docx text extraction

In [ ]:
from extraction.pipeline import process_resume

pdf_text = process_resume("Downloads/My_Resume(9).pdf")
docx_text = process_resume("Downloads/Onyeiwu_Gabriel_Chibuzor_Graduate_Trainee_CV.docx")

print("PDF result:", len(pdf_text), "characters")
print("DOCX result:", len(docx_text), "characters")

In [ ]:
try:
    process_resume("Downloads/some_file.txt")
except ValueError as e:
    print("Correctly caught error:", e)

# STAGE 2

### Create the CV and Job Aalyser

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

print("Key loaded:", api_key is not None)

In [ ]:
import json, os
from extraction.pipeline import process_resume
from Ai_analyser.cv_analyzer import analyze_cv
from Ai_analyser.jd_analyzer import analyze_jd

os.makedirs("test_data", exist_ok=True)

# ONE canonical CV
cv_text = process_resume("Downloads/test-resume.pdf")
result_cv = analyze_cv(cv_text)

# ONE canonical JD
jd_text = """
Full job description
Job summary

We are looking for a skilled and motivated Software Engineer to join our growing technology team. The ideal candidate will be responsible for designing, developing, testing, deploying, and maintaining reliable and scalable software solutions.

Min Qualification: Degree Experience Level: Mid level Experience Length: 4 years Language Requirement: English Working Hours: Contract - Flexible Hours Applicant Location: Lagos, Nigeria
Job descriptions & requirements

Responsibilities:


Design, develop, test, and maintain software applications and features.
Translate business and product requirements into effective technical solutions.
Write clean, efficient, scalable, and maintainable code.
Develop and integrate RESTful APIs and third-party services.
Work with databases and ensure data integrity and performance.
Identify, troubleshoot, and resolve bugs and production issues.
Participate in code reviews and follow software development best practices.
Collaborate with product managers, designers, and other developers.
Contribute to system architecture and technical decision-making.
Write and maintain technical documentation.
Participate in testing, deployment, and continuous improvement of applications.
Stay updated with emerging technologies and recommend improvements where appropriate.

Requirements:

Bachelor's degree in Computer Science, Software Engineering, Computer Engineering, Information Technology, or a related field is preferred.
2–5 years of professional software development experience.
Strong proficiency in at least one programming language such as JavaScript/TypeScript, Python, Java, C#, PHP, or Go.
Experience with modern frameworks such as React, Next.js, Node.js, Django, Laravel, Spring Boot, or equivalent.
Good understanding of databases such as PostgreSQL, MySQL, MongoDB, or Redis.
Experience building and consuming RESTful APIs.
Proficiency with Git and version-control workflows.
Good understanding of software development principles, testing, debugging, and deployment.
Strong problem-solving and analytical skills.
Ability to work independently in a remote environment.
Strong communication and teamwork skills.

Preferred Qualifications:

Experience with cloud platforms such as AWS, Azure, or Google Cloud.
Familiarity with Docker and CI/CD pipelines.
Experience with microservices or distributed systems.
Knowledge of automated testing.
Experience working in an Agile/Scrum environment.
Familiarity with DevOps practices is an added advantage.

Benefits:

Fully remote work environment.
Opportunity to work on meaningful and challenging technology projects.
Collaborative and growth-focused team.
Opportunities for professional development and career growth.

Location: Remote.

Remuneration: NGN 300,000 – 700,000
"""

result_jd = analyze_jd(jd_text)

with open("test_data/cv_math_teacher.json", "w") as f:
    json.dump(result_cv, f, indent=2)
with open("test_data/jd_accountant.json", "w") as f:
    json.dump(result_jd, f, indent=2)

print("CV skills saved:", len(result_cv['skills']))
print("JD required_skills saved:", len(result_jd['required_skills']))

In [ ]:
# URL extraction of JOB description

'''from extraction.url_reader import extract_text_from_url

url = "https://jobs.accaglobal.com/job/14101931/accountant/#:~:text=Harcourt...Next%20job-,Accountant,-Employer"
jd_text = extract_text_from_url(url)
print(jd_text[:1000])
print(f"\n\nTotal characters extracted: {len(jd_text)}")
'''


# STAGE 3

### Matching the CV ana Job similarities

In [ ]:
import json

print("=" * 50)
print("CHECKING test_data/cv_math_teacher.json")
print("=" * 50)
with open("test_data/cv_math_teacher.json") as f:
    cv_data = json.load(f)
print("Number of CV skills:", len(cv_data.get("skills", [])))
print("First 5 CV skills:", cv_data.get("skills", [])[:5])

print()
print("=" * 50)
print("CHECKING test_data/jd_accountant.json")
print("=" * 50)
with open("test_data/jd_accountant.json") as f:
    jd_data = json.load(f)
print("Number of JD required_skills:", len(jd_data.get("required_skills", [])))
print("First 5 JD required_skills:", jd_data.get("required_skills", [])[:5])

In [ ]:
# We used the embedding martching parttern to check for the similarity between word from the CV and Job.
# the funtion return the scores of the similarities phrases and the unrelated phrases.

from matching.semantic_matcher import semantic_similarity

score1 = semantic_similarity("predictive modeling", "machine learning forecasting")
score2 = semantic_similarity("predictive modeling", "gardening")  # should be low

print(f"Related phrases:, {score1:.2f}")
print(f"Unrelated phrases:,{score2:.2f}")

In [ ]:
import json

with open("test_data/cv_math_teacher.json") as f:
    result_cv = json.load(f)

with open("test_data/jd_accountant.json") as f:
    result_jd = json.load(f)

from matching.exact_matcher import exact_skill_match
from matching.semantic_matcher import semantic_skill_match

exact_result = exact_skill_match(result_cv['skills'], result_jd['required_skills'])
semantic_result = semantic_skill_match(result_cv['skills'], result_jd['required_skills'])

print("EXACT MATCH:")
print(exact_result)
print()
print("SEMANTIC MATCH:")
print(semantic_result)

# STAGE 4

### Create a keyword search(Using Job Keyword to search the CV)

In [ ]:
import json
from extraction.pipeline import process_resume
from matching.keyword_matcher import keyword_match

with open("test_data/jd_accountant.json") as f:
    result_jd = json.load(f)

# re-extract the raw CV text (not the structured skills)
cv_raw_text = process_resume("Downloads/test-resume.pdf")

kw_result = keyword_match(cv_raw_text, result_jd.get('keywords', []))
print(kw_result)

# STAGE 5

### Building the ATS scoring Board

#### Adding  exact/semantic, keyword score, Experence score, Education score, Responsibilite score and Formatting to the ATS system

In [ ]:
#Verifying if the logic works
# Runing a pipeline that combine the follow
# upload of cv/job
# get the exact/semantic, keyword score, experience score, educational score, Responsibilitie score and Formatting score
# ats scoring system at once.

from pipeline_runner import run_full_analysis

jd_text = """
Full job description
Job summary

We are looking for a skilled and motivated Software Engineer to join our growing technology team. The ideal candidate will be responsible for designing, developing, testing, deploying, and maintaining reliable and scalable software solutions.

Min Qualification: Degree Experience Level: Mid level Experience Length: 4 years Language Requirement: English Working Hours: Contract - Flexible Hours Applicant Location: Lagos, Nigeria
Job descriptions & requirements

Responsibilities:


Design, develop, test, and maintain software applications and features.
Translate business and product requirements into effective technical solutions.
Write clean, efficient, scalable, and maintainable code.
Develop and integrate RESTful APIs and third-party services.
Work with databases and ensure data integrity and performance.
Identify, troubleshoot, and resolve bugs and production issues.
Participate in code reviews and follow software development best practices.
Collaborate with product managers, designers, and other developers.
Contribute to system architecture and technical decision-making.
Write and maintain technical documentation.
Participate in testing, deployment, and continuous improvement of applications.
Stay updated with emerging technologies and recommend improvements where appropriate.

Requirements:

Bachelor's degree in Computer Science, Software Engineering, Computer Engineering, Information Technology, or a related field is preferred.
2–5 years of professional software development experience.
Strong proficiency in at least one programming language such as JavaScript/TypeScript, Python, Java, C#, PHP, or Go.
Experience with modern frameworks such as React, Next.js, Node.js, Django, Laravel, Spring Boot, or equivalent.
Good understanding of databases such as PostgreSQL, MySQL, MongoDB, or Redis.
Experience building and consuming RESTful APIs.
Proficiency with Git and version-control workflows.
Good understanding of software development principles, testing, debugging, and deployment.
Strong problem-solving and analytical skills.
Ability to work independently in a remote environment.
Strong communication and teamwork skills.

Preferred Qualifications:

Experience with cloud platforms such as AWS, Azure, or Google Cloud.
Familiarity with Docker and CI/CD pipelines.
Experience with microservices or distributed systems.
Knowledge of automated testing.
Experience working in an Agile/Scrum environment.
Familiarity with DevOps practices is an added advantage.

Benefits:

Fully remote work environment.
Opportunity to work on meaningful and challenging technology projects.
Collaborative and growth-focused team.
Opportunities for professional development and career growth.

Location: Remote.

Remuneration: NGN 300,000 – 700,000
"""

result = run_full_analysis("Downloads/test-resume.pdf", jd_text)
print(result['score'])

# STAGE 6

### Tailoring the Users CV to match the JD. that let the LLM model recreate your CV

In [ ]:
# Recreating the CV using the LLM model
from tailoring.tailor import tailor_summary

# reuse result_cv / result_jd from your SWE test pair
new_summary = tailor_summary(result_cv, result_jd)
print(new_summary)

In [ ]:
# Verifying the recreated CV
from tailoring.verify import verify_summary

# reuse the CV data and the summary from your last run
check = verify_summary(result_cv, new_summary)
print(check)

In [ ]:
# Combining the two recreating and verifying the CV
from tailoring.tailor_pipeline import generate_verified_summary

result = generate_verified_summary(result_cv, result_jd)
print("Recommendation:", result['recommendation'])
print()
print("Tailored summary:")
print(result['tailored_summary'])
print()
print("Verification details:", result['verification'])

In [ ]:
# Building experience tailoring for the recreating the CV and also verify it.
from tailoring.tailor_experience import tailor_experience
from tailoring.verify import verify_summary

bullets = tailor_experience(result_cv, result_jd)
print("Bullets:", bullets)

joined_text = "\n".join(bullets)
check = verify_summary(result_cv, joined_text)
print("Verification:", check)

In [ ]:
#Building skills tailoring for the recreating the CV and also verify it.
from tailoring.tailor_skills import tailor_skills
from tailoring.verify import verify_skills_reorder

reordered = tailor_skills(result_cv['skills'], result_jd['required_skills'])
print("Reordered:", reordered)

check = verify_skills_reorder(result_cv['skills'], reordered)
print("Verification:", check)

In [ ]:
print("Original CV skills (this run):", result_cv['skills'])

# STAGE 7

### Interview Question Generator Using the CV and JD

In [ ]:
# interview generator
from interview.question_generator import generate_interview_questions

questions = generate_interview_questions(result_cv, result_jd)
print(json.dumps(questions, indent=2))

In [ ]:
from tailoring.verify import verify_summary

cv_questions_text = "\n".join(questions['cv_based_questions'])
check = verify_summary(result_cv, cv_questions_text)
print(check)

In [ ]:
from tailoring.verify import verify_summary

check = verify_summary(result_cv, cv_questions_text)
print(check)

In [ ]:
print(cv_questions_text)

In [ ]:
from interview.interview_pipeline import generate_verified_questions

result = generate_verified_questions(result_cv, result_jd)
print(json.dumps(result, indent=2))

# STAGE 8

### New CV document generator

In [ ]:
from generation.docx_builder import build_tailored_resume

path = build_tailored_resume(
    candidate_name="Your Name Here",
    tailored_summary=new_summary,
    reordered_skills=reordered,
    tailored_bullets=bullets,
    experience=result_cv['experience'],
    education=result_cv['education'],
    certifications=result_cv['certifications'],
    cv_text=cv_text,
    output_path="tailored_resume.docx"
)
print("Saved to:", path)